In [8]:
import os
os.environ['MPLCONFIGDIR'] = "/mnt/storage6/grace/plt_temp/"
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import xarray as xr
import netCDF4 as nc
from netCDF4 import Dataset
import scipy as sp

import datetime
import cftime
import time

import gsw

In [9]:
ds_t4 = xr.open_mfdataset("/mnt/storage6/myers/NEMO/ANHA4-EPM151/ANHA4-EPM151_y2018m05d25_gridT.nc")
ds_t12 = xr.open_mfdataset("/mnt/storage6/myers/NEMO/ANHA12-EXH006/ANHA12-EXH006_y2018m05d25_gridT.nc")

sal4 = ds_t4['vosaline'][0, :, 250:450, 150:375]
sal12 = ds_t12['vosaline'][0, :, 750:1350, 450:1125]

temp4 = ds_t4['votemper'][0, :, 250:450, 150:375]
temp12 = ds_t12['votemper'][0, :, 750:1350, 450:1125]

s4 = gsw.SA_from_Sstar(sal4 * (35.16504/35), 10.1325, ds_t4.nav_lon_grid_T[250:450, 150:375], ds_t4.nav_lat_grid_T[250:450, 150:375]) #absolute salinity
s12 = gsw.SA_from_Sstar(sal12 * (35.16504/35), 10.1325, ds_t12.nav_lon[750:1350, 450:1125], ds_t12.nav_lat[750:1350, 450:1125]) #absolute salinity


In [10]:
sal4 = ds_t4['vosaline']
temp4 = ds_t4['votemper']
s4 = gsw.SA_from_Sstar(sal4 * (35.16504/35), 10.1325, ds_t4.nav_lon_grid_T, ds_t4.nav_lat_grid_T)
rho4 = gsw.sigma0(s4, temp4)[0,:,250:450, 150:375]

In [13]:
rho_goal = 27.68
residuals = rho4 - rho_goal
resid4 = np.abs(rho4 - rho_goal)
print(resid4.shape)
#resid12 = np.abs(rho12 - rho_goal)
b4 = np.stack([np.nanmin(resid4, axis=0)]*50, axis=0)
#b12 = np.nanmin(resid12, axis=0)

(50, 200, 225)


/tmp/ipykernel_621077/3951358814.py:6: RuntimeWarning: All-NaN axis encountered
  b4 = np.stack([np.nanmin(resid4, axis=0)]*50, axis=0)


In [20]:
residuals.shape

(50, 200, 225)

In [41]:
zerodepth = np.zeros((200, 225))
onedepth = np.zeros((200, 225))
rho = np.zeros((50, 200, 225))
rho1 = np.zeros((50, 200, 225))

depth = sal4.deptht.compute()
new_row = np.ones((200, 225))
updated_matrix = np.insert(residuals, 0, new_row, axis=0)

In [42]:
updated_matrix[:-1,:,:].shape

(50, 200, 225)

In [49]:
(updated_matrix[:,:,:]).shape

(51, 200, 225)

In [44]:
rho4.shape

(50, 200, 225)

In [51]:
(updated_matrix[:,:,:] * updated_matrix[:,:,:]).shape

ValueError: conflicting sizes for dimension 'deptht': length 51 on <this-array> and length 50 on {'y_grid_T': 'nav_lat_grid_T', 'x_grid_T': 'nav_lat_grid_T', 'deptht': 'deptht'}

In [47]:
rhothing = np.where(updated_matrix[:-1,:,:] * updated_matrix[1:,:,:] < 0, rho4, 0)


ValueError: conflicting sizes for dimension 'deptht': length 50 on <this-array> and length 49 on {'y_grid_T': 'nav_lat_grid_T', 'x_grid_T': 'nav_lat_grid_T', 'deptht': 'deptht'}

In [29]:
rhothing.shape

AttributeError: 'tuple' object has no attribute 'shape'

In [ ]:
for x in range(0,49):
    change = np.where(residuals[x,:,:] * residuals[x+1,:,:] < 0)
    zerodepth[change] = depth[x]
    onedepth[change] = depth[x+1]

In [ ]:
m = np.unique(zeroinds).astype(np.int64)
n = np.unique(oneinds).astype((np.int64)

In [ ]:
ds_t4[0,:,:,:]

In [23]:
change = np.where(np.sign([-1,4,5,-3]) != np.sign([-3,4,5,4]))
change

(array([3]),)

In [ ]:
rho4.isel(deptht=zeroinds)

In [ ]:
n[zeroinds][:,:,0,0]

In [ ]:
plt.contourf(zeroinds, np.arange(0,50,1))
plt.colorbar()

In [ ]:
negprev

In [ ]:
l = np.where(residuals < 0)
low[l] = np.interp(0, [residuals[x-1,:,:][v], residuals[x,:,:][v]], [rho4.depth[x-1], rho_4.depth[x]])

h = np.where(residuals > 0)
if x == 49:
    low[h] = rho4.depth[49]
else:
    low[h] = np.interp(0, [residuals[x,:,:][v], residuals[x + 1,:,:][v]], [rho4.depth[x], rho_4.depth[x+1]])

In [ ]:
low = np.zeros((200, 225))
low.shape

In [ ]:
from scipy.interpolate import make_interp_spline
spl = make_interp_spline(ds_t4.deptht, residuals, k=1)

In [ ]:
spl([0])

In [ ]:
l = np.where(resid4 == b4)
l

In [ ]:

        
        





                                   
"""

    for y in range(len(v[0])):
        if resid4[x,v[0][y], v[1][y]] < 0: # water is lighter than the goal rho
            
            low[v[0][y], v[1][y]] = np.interp(0, [resid4[x, v[0][y], v[1][y]], resid4[x +1, v[0][y], v[1][y]]], [rho_4.depth[x], rho_4.depth[x]])

        else: #water is denser than the goal rho
            if x == 0: # can occur with outcropping
                low[v[0][y], v[1][y]] = 0
            else:
                low[v[0][y], v[1][y]] = np.interp(0, [resid4[x, v[0][y], v[1][y]], resid4[x - 1, v[0][y], v[1][y]]], [rho_4.depth[x], rho_4.depth[x]])
"""
            

"""
    if len(v[0]) > 0:
        print(v)
        print(low)
    #print(v)
    #m = np.where(resid4[x,:,:][v] < 0) #under
    #print(m)
    #p = np.where(resid4[x,:,:][v] > 0) #over
    #print(v)

    #low[m] = x
    #high[m] = x+1

    #low[p] = x-1
    #high[p] = x
"""
    

In [ ]:
np.min(low)